In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch

In [2]:
#read csv: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
veri = pd.read_csv("IMDB Dataset.csv")


In [3]:
veri.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print(veri.columns)
print(veri.shape)
print(veri["sentiment"].value_counts())
veri_kucuk = veri.sample(n=2000, random_state=42).reset_index(drop=True)

Index(['review', 'sentiment', 'review_clean'], dtype='str')
(50000, 3)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Index(['review', 'sentiment', 'review_clean'], dtype='str')


In [26]:
veri_kucuk["label"] = veri_kucuk["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [27]:
veri_kucuk[["review_clean", "sentiment", "label"]].head()





,review_clean,sentiment,label
0,I really liked this Summerslam due to the look...,positive,1
1,Not many television shows appeal to quite as m...,positive,1
2,The film quickly gets to a major chase scene w...,negative,0
3,Jane Austen would definitely approve of this o...,positive,1
4,Expectations were somewhat high for me when I ...,negative,0


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

#tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
#model = AutoModel.from_pretrained("bert-base-multilingual-cased")

#model.eval()



tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

In [8]:
def bert_embedding(metin):
    inputs = tokenizer(
        metin,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Her token için 768 boyutlu vektör var
    token_embeddings = outputs.last_hidden_state

    # Tokenların ortalamasını alıp
    # bütün yorumu tek 768 boyutlu vektöre indiriyoruz
    sentence_embedding = token_embeddings.mean(dim=1)

    return sentence_embedding.squeeze().numpy()

In [29]:
embedding_listesi = []

for i, yorum in enumerate(veri_kucuk["review_clean"]):
    embedding = bert_embedding(yorum)
    embedding_listesi.append(embedding)

    if (i + 1) % 50 == 0:
        print(i + 1, "yorum tamamlandı")

50 yorum tamamlandı
100 yorum tamamlandı
150 yorum tamamlandı
200 yorum tamamlandı
250 yorum tamamlandı
300 yorum tamamlandı
350 yorum tamamlandı
400 yorum tamamlandı
450 yorum tamamlandı
500 yorum tamamlandı
550 yorum tamamlandı
600 yorum tamamlandı
650 yorum tamamlandı
700 yorum tamamlandı
750 yorum tamamlandı
800 yorum tamamlandı
850 yorum tamamlandı
900 yorum tamamlandı
950 yorum tamamlandı
1000 yorum tamamlandı
1050 yorum tamamlandı
1100 yorum tamamlandı
1150 yorum tamamlandı
1200 yorum tamamlandı
1250 yorum tamamlandı
1300 yorum tamamlandı
1350 yorum tamamlandı
1400 yorum tamamlandı
1450 yorum tamamlandı
1500 yorum tamamlandı
1550 yorum tamamlandı
1600 yorum tamamlandı
1650 yorum tamamlandı
1700 yorum tamamlandı
1750 yorum tamamlandı
1800 yorum tamamlandı
1850 yorum tamamlandı
1900 yorum tamamlandı
1950 yorum tamamlandı
2000 yorum tamamlandı


In [34]:
X = np.array(embedding_listesi)

y = veri_kucuk["label"].values

print(X.shape)
print(y.shape)

(2000, 768)
(2000,)


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=44,
    stratify=y
)

model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7775
              precision    recall  f1-score   support

           0       0.76      0.82      0.79       205
           1       0.80      0.73      0.76       195

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.78       400
weighted avg       0.78      0.78      0.78       400



In [32]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

model_svm = SVC(kernel="linear")

model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.7775
              precision    recall  f1-score   support

           0       0.76      0.83      0.79       205
           1       0.80      0.72      0.76       195

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.78       400
weighted avg       0.78      0.78      0.78       400



In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model_rf = RandomForestClassifier(
    n_estimators=350,
    random_state=44,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.7725
              precision    recall  f1-score   support

           0       0.76      0.82      0.79       205
           1       0.79      0.72      0.76       195

    accuracy                           0.77       400
   macro avg       0.77      0.77      0.77       400
weighted avg       0.77      0.77      0.77       400

